In [5]:
import pandas as pd
import torch
from torchvision import models
from torchvision.models import resnet18
import torch.nn as nn
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F
import os
import cv2
from PIL import Image


## Step 1: Load model

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [7]:
# Load the initial model
NUM_CLASSES = 7

model = resnet18(weights=None)
model.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(model.fc.in_features, 7))

In [8]:
MODEL_PATH = r"S:\Projects\emotion_detection\artifacts\model_trainer\best_rafdb_resnet18.pth"

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Step 2 : inference preprocessing

In [9]:
# Introduce inference tansform, same as during training
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [10]:
# Get test image
image = Image.open(r"S:\Projects\emotion_detection\artifacts\data_ingestion\Organized\train\happy\train_00036_aligned.jpg")

In [11]:
# apply transform
image_tensor = inference_transform(image)

In [12]:
print(image_tensor.shape)

torch.Size([3, 224, 224])


In [13]:
# add batch
image_tensor = image_tensor.unsqueeze(0)

In [14]:
print(image_tensor.shape)

torch.Size([1, 3, 224, 224])


In [15]:
image_tensor = image_tensor.to(device)

## Step 3: single image prediction

In [16]:
with torch.no_grad():
    outputs = model(image_tensor)

print(outputs.shape)    

torch.Size([1, 7])


In [17]:
# Convert logits into probabilities
probabilities = F.softmax(outputs, dim=1)
print(probabilities)

tensor([[0.0320, 0.0290, 0.0037, 0.2499, 0.4894, 0.1593, 0.0366]],
       device='cuda:0')


In [18]:
# Get highest probability
confidence, predicted = torch.max(probabilities, dim=1)

In [19]:
# Label map
class_names = [
    'angry', 
    'disgust', 
    'fear', 
    'happy', 
    'neutral', 
    'sad', 
    'surprise'
]

In [20]:
emotion = class_names[predicted.item()]

print(f"Prediction : {emotion}")
print(f"Confidence : {confidence.item()*100:.2f}%")

Prediction : neutral
Confidence : 48.94%


In [21]:
# helper function for testing
class_names = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]


def predict_emotion(
    model,
    transform,
    device,
    image_path=None,
    frame=None
):
    """
    Predict emotion from a single image.

    Args:
        image_path (str): Path to image.
        model: Trained PyTorch model.
        transform: Inference transform.
        device: cuda or cpu.

    Returns:
        predicted_label, confidence
    """

    # Load image
    if image_path is not None:

        image = Image.open(image_path).convert("RGB")

    elif frame is not None:

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)

    else:
        raise ValueError("Provide either image_path or frame.")

    # Preprocess
    image = transform(image)

    # Add batch dimension
    image = image.unsqueeze(0).to(device)

    # Prediction
    model.eval()
    with torch.no_grad():
        outputs = model(image)

        probabilities = F.softmax(outputs, dim=1)

        confidence, predicted = torch.max(probabilities, dim=1)

    predicted_label = class_names[predicted.item()]
    confidence = confidence.item() * 100

    # Convert probabilities into a dictionary
    all_probabilities = {
        label: prob * 100
        for label, prob in zip(
            class_names,
            probabilities.squeeze().cpu().numpy()
        )
    }

    return predicted_label, confidence, all_probabilities

In [22]:
prediction, confidence, probabilities = predict_emotion(
    image_path=r"S:\Projects\emotion_detection\artifacts\data_ingestion\Organized\train\sad\train_00692_aligned.jpg",
    model=model,
    transform=inference_transform,
    device=device
)

print(f"\nPrediction : {prediction}")
print(f"Confidence : {confidence:.2f}%\n")

print("Class Probabilities:")
for emotion, prob in probabilities.items():
    print(f"{emotion:<10}: {prob:.2f}%")


Prediction : neutral
Confidence : 67.63%

Class Probabilities:
angry     : 0.00%
disgust   : 1.84%
fear      : 0.00%
happy     : 0.16%
neutral   : 67.63%
sad       : 30.37%
surprise  : 0.00%


### Batch inference

In [23]:
def batch_predict(folder_path, model, transform, device):
    """
    Predict emotions for all images in a folder.

    Args:
        folder_path (str): Folder containing images.
        model: Trained PyTorch model.
        transform: Inference transform.
        device: cpu or cuda.

    Returns:
        List of prediction results.
    """

    results = []

    supported_extensions = (".jpg", ".jpeg", ".png", ".bmp") 

    for image_name in os.listdir(folder_path):

        if not image_name.lower().endswith(supported_extensions):
            continue

        image_path = os.path.join(folder_path, image_name)

        prediction, confidence, probabilities = predict_emotion(
            image_path=image_path,
            model=model,
            transform=transform,
            device=device
        )

        results.append({
            "image": image_name,
            "prediction": prediction,
            "confidence": confidence,
            "probabilities": probabilities
        })

    return results

In [24]:
results = batch_predict(
    folder_path=r"S:\Projects\emotion_detection\test_images",
    model=model,
    transform=inference_transform,
    device=device
)

In [25]:
for result in results:

    print("-" * 50)

    print(f"Image      : {result['image']}")
    print(f"Prediction : {result['prediction']}")
    print(f"Confidence : {result['confidence']:.2f}%")

    print("\nClass Probabilities:")

    for emotion, prob in result["probabilities"].items():
        print(f"{emotion:<10}: {prob:.2f}%")

--------------------------------------------------
Image      : happy_1.jpg
Prediction : neutral
Confidence : 44.21%

Class Probabilities:
angry     : 0.09%
disgust   : 1.30%
fear      : 0.05%
happy     : 0.52%
neutral   : 44.21%
sad       : 12.40%
surprise  : 41.43%
--------------------------------------------------
Image      : happy_2.jpg
Prediction : neutral
Confidence : 83.27%

Class Probabilities:
angry     : 0.63%
disgust   : 0.04%
fear      : 0.00%
happy     : 0.50%
neutral   : 83.27%
sad       : 1.59%
surprise  : 13.97%
--------------------------------------------------
Image      : neutral_1.jpg
Prediction : neutral
Confidence : 62.36%

Class Probabilities:
angry     : 0.35%
disgust   : 0.62%
fear      : 0.04%
happy     : 0.03%
neutral   : 62.36%
sad       : 1.92%
surprise  : 34.70%
--------------------------------------------------
Image      : neutral_2.jpg
Prediction : neutral
Confidence : 79.68%

Class Probabilities:
angry     : 0.01%
disgust   : 0.06%
fear      : 0.01%
h

In [28]:
predict_emotion(
    image_path=r"S:\Projects\emotion_detection\artifacts\data_ingestion\Organized\test\sad\test_0006_aligned.jpg",
    model=model,
    transform=inference_transform,
    device=device
)

('sad',
 83.36901664733887,
 {'angry': 0.00018318692127650138,
  'disgust': 0.18544396152719855,
  'fear': 0.0003669303168862825,
  'happy': 0.008638873259769753,
  'neutral': 16.424116492271423,
  'sad': 83.36901664733887,
  'surprise': 0.012238847557455301})

### Real time inference

In [29]:
# Load pretrained face detector
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

In [32]:
# Replace webcam loop with face detector
cap = cv2.VideoCapture(0)

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_detector.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(100, 100)
    )

    for (x, y, w, h) in faces:

        face = frame[y:y+h, x:x+w]

        prediction, confidence, _ = predict_emotion(
            model=model,
            transform=inference_transform,
            device=device,
            frame=face
        )

        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        label = f"{prediction} ({confidence:.1f}%)"

        cv2.putText(
            frame,
            label,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

    cv2.imshow("Emotion Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()